In [ ]:
import sys
sys.path.append("..")

import glob
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision.transforms as T

from PIL import Image
from collections import deque
from torchmetrics import Metric

from src.metrics import SegIoU, WarpedTC, VEC
from src.flow import warp
import random


In [ ]:
p1 = "/data/video-segmentation/cityscapes/leftImg8bit_sequence/train/aachen/aachen_000000_000019_leftImg8bit.png"
p2 = "/data/video-segmentation/cityscapes/leftImg8bit_sequence/train/aachen/aachen_000000_000023_leftImg8bit.png"
label_path = "/data/video-segmentation/cityscapes/gtFine/train/aachen/aachen_000000_000019_gtFine_labelTrainIds.png"


img1 = Image.open(p1)
img2 = Image.open(p2)
img1

In [ ]:
test1 = torch.randint(0, 5, (3, 17, 23))
test2 = torch.randint(0, 5, (3, 17, 23))

test1[0, 0, 0] = 255

In [ ]:
seg_iou = SegIoU(5, 255)
seg_iou.update(test2, test1)
seg_iou.compute()

In [ ]:
from torchvision.models.optical_flow import raft_large, Raft_Large_Weights

weights = Raft_Large_Weights.DEFAULT
raft = raft_large(weights=weights).eval().cuda()
raft_tf = weights.transforms()

@torch.inference_mode()
def compute_flow(src, dst):
    """Flow src -> dst. src, dst: (B,3,H,W) float in [0,1]; H, W divisible by 8."""
    a, b = raft_tf(src, dst)
    return raft(a, b)[-1]

a, b = raft_tf(img1.reduce(2), img2.reduce(2))
a = a.unsqueeze(0).to("cuda")
b = b.unsqueeze(0).to("cuda")

# size = 128
# y, x = torch.meshgrid(torch.arange(size), torch.arange(size))
# pos1 = (torch.stack((x, y), dim=-1) - size / 2) / torch.tensor(4)
# pos2 = (torch.stack((x, y), dim=-1) - size / 2) / torch.tensor(2)
# a = torch.exp(-0.5 * torch.square(pos1).sum(-1))
# b = torch.exp(-0.5 * torch.square(pos2).sum(-1))
# a = a[None, None, :].expand(-1, 3, -1, -1).to("cuda")
# b = b[None, None, :].expand(-1, 3, -1, -1).to("cuda")

with torch.no_grad():
    flow_fw = raft(a, b)[-1]
    flow_bw = raft(b, a)[-1]

In [ ]:
pred_t = torch.randint(0, 5, b[:, 0].shape).to("cuda")
pred_prev = torch.randint(0, 5, a[:, 0].shape).to("cuda")

warped_tc = WarpedTC(5).to("cuda")
warped_tc.update(pred_t, pred_prev, flow_bw, flow_fw)
warped_tc.compute()

In [ ]:
# tc = WarpedTC().cuda()

# @torch.inference_mode()
# def eval_snippet(frames):  # frames: (T,3,H,W) float in [0,1] on cuda
#     preds = torch.cat([to_pred(model(f[None]), f.shape[-2:]) for f in frames])
#     for t in range(1, len(frames)):
#         prev, cur = frames[t - 1 : t], frames[t : t + 1]
#         flow_bw = compute_flow(cur, prev)   # t -> t-1
#         flow_fw = compute_flow(prev, cur)   # t-1 -> t
#         tc.update(preds[t : t + 1], preds[t - 1 : t], flow_bw, flow_fw)

# # for frames in snippets: eval_snippet(frames)
# # print(tc.compute())

# torch.manual_seed(0)
# B, H, W = 1, 128, 256
# p = F.interpolate(torch.randint(0, 19, (B, 1, 8, 16)).float(), size=(H, W)).squeeze(1).long()
# prev = torch.roll(p, 4, dims=-1)                      # objects were 4 px to the right at t-1
# flow_bw = torch.zeros(B, 2, H, W); flow_bw[:, 0] = 4  # t -> t-1: +4 in x
# flow_fw = -flow_bw

# m = WarpedTC(); m.update(p, prev, flow_bw, flow_fw)
# print("correct flow:", m.compute())                   # TC = 1.0, coverage < 1 (right edge masked)

# m = WarpedTC(); z = torch.zeros_like(flow_bw); m.update(p, prev, z, z)
# print("zero flow:   ", m.compute())                   # TC < 1

In [ ]:
labels = [torch.randint(0, 5, (256, 512)) for _ in range(10)]
pred = [torch.randint(0, 5, (256, 512)) for _ in range(10)]

for i in range(10):
    for _ in range(5):
        h = random.randint(0, 255)
        w = random.randint(0, 511)
        labels[i][h, w] = 255


In [ ]:
vec = VEC(5, 8, 255)

for i in range(10):
    vec.update(pred[i], labels[i])
    print(vec.compute())